<!-- Banner Image -->
<center>
    <img src="https://developer-blogs.nvidia.com/wp-content/uploads/2024/07/rag-representation.jpg" width="75%">
</center>

<!-- Links -->
<center>
  <a href="https://www.nvidia.com/en-us/deep-learning-ai/solutions/data-science/workbench/" style="color: #76B900;">NVIDIA AI Workbench</a> •
  <a href="https://docs.nvidia.com/ai-workbench/" style="color: #76B900;">User Documentation</a> •
  <a href="https://docs.nvidia.com/ai-workbench/user-guide/latest/quickstart/example-projects.html" style="color: #76B900;">Example Projects Catalog</a> •
  <a href="https://forums.developer.nvidia.com/t/support-workbench-example-project-llama-3-finetune/303411" style="color: #76B900;"> Problem? Submit a ticket here! </a>
</center>

# Finetune Llama-3.3-8B-Instruct on DGX Spark GB10 using DPO

Welcome!

This notebook is optimized for **NVIDIA DGX Spark with GB10** (Grace Blackwell architecture). With 128GB of unified memory, we can perform **full-precision BF16 training** without any quantization.

### Key Features for DGX Spark GB10:
- **No quantization needed** - 128GB unified memory handles full BF16 training easily
- **No bitsandbytes required** - We don't need 4-bit quantization
- **Native SDPA attention** - Uses PyTorch's Scaled Dot Product Attention (no Flash Attention required)
- **Larger batch sizes** - More memory allows for faster training

### What is Direct Preference Optimization (DPO)?

DPO improves on RLHF by treating alignment as a classification problem. It uses the trained model and a reference model copy. During training, the goal is to make the trained model output higher probabilities for preferred answers and lower probabilities for rejected answers. Because the LLM uses itself as a reward model, it aligns without needing a separate reward model.

#### Help us make this tutorial better! Please provide feedback on the [NVIDIA Developer Forum](https://forums.developer.nvidia.com/c/ai-data-science/nvidia-ai-workbench/671).

## Table of Contents
1. Verify GPU and Environment
2. Import Libraries
3. Load Model and Dataset
4. Configure DPO Training
5. Train with DPO
6. Save and Test Model

## 1. Verify GPU and Environment

Let's verify we're running on the DGX Spark GB10.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

## 2. Import Libraries

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2024 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os
import gc
import torch
import transformers
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments,
)
from datasets import load_dataset
from peft import (
    LoraConfig, 
    PeftModel, 
    prepare_model_for_kbit_training
)
from trl import DPOTrainer

## 3. Load Model and Dataset

We load the model in **BF16 precision** without any quantization. The DGX Spark GB10's 128GB unified memory easily handles both the model and reference model.

**Important:** We use `attn_implementation="sdpa"` (Scaled Dot Product Attention) instead of Flash Attention for Blackwell compatibility.

In [ ]:
# Model configuration
base_model = "allura-forge/Llama-3.3-8B-Instruct"
new_model = "/project/models/NV-DPOLlama-3.3-8B"

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, token=os.environ["HF_KEY"])

# Load model in BF16 - NO QUANTIZATION needed on DGX Spark GB10!
# Using SDPA instead of Flash Attention for Blackwell compatibility
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    token=os.environ["HF_KEY"], 
    cache_dir="/project/models",
    torch_dtype=torch.bfloat16,  # Full BF16 precision
    attn_implementation="sdpa",   # Use native SDPA instead of Flash Attention
    device_map="auto",
)

# Reference model - also in BF16
ref_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    token=os.environ["HF_KEY"], 
    cache_dir="/project/models",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",
)

print(f"Model loaded in BF16 with SDPA attention")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# LoRA configuration - can use higher rank on GB10 due to more memory
peft_config = LoraConfig(
    r=32,  # Increased from 16 for better quality on GB10
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

## Load and Format Dataset

Preference datasets contain the original prompt along with `chosen` and `rejected` columns.

In [ ]:
# Load dataset
dataset = load_dataset("Intel/orca_dpo_pairs")['train']

# Use a subset for demonstration - increase for full training
dataset = dataset.shuffle(seed=42).select(range(500))  # Increased from 150 for GB10
print(f"Dataset loaded: {len(dataset)} samples")

In [ ]:
# Explore the dataset
print("Example question:")
print(dataset[19]['question'][:200] + "...")

In [ ]:
# Load and apply chat template
chat_template = open('chat_template/llama-3-instruct.jinja').read()
chat_template = chat_template.replace('    ', '').replace('\n', '')
tokenizer.chat_template = chat_template

In [ ]:
def dataset_format(example):
    # Format system
    if len(example['system']) > 0:
        message = {"role": "system", "content": example['system']}
        system = tokenizer.apply_chat_template([message], tokenize=False)
    else:
        system = ""
    # Format instruction
    message = {"role": "user", "content": example['question']}
    prompt = tokenizer.apply_chat_template([message], tokenize=False, add_generation_prompt=True)
    # Format chosen answer
    chosen = example['chosen'] + "<|eot_id|>\n"
    # Format rejected answer
    rejected = example['rejected'] + "<|eot_id|>\n"
    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

In [ ]:
original_columns = dataset.column_names
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dataset = dataset.map(
    dataset_format,
    remove_columns=original_columns,
    num_proc=os.cpu_count(),
)
print("Dataset formatted with Llama 3 chat template")

## 4. Configure DPO Training

Training arguments optimized for DGX Spark GB10's 128GB memory.

In [ ]:
### Uncomment to use Weights and Biases ###

# import wandb
# wandb.login()

In [ ]:
# DGX Spark GB10 optimized training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=4,   # Increased from 2 for GB10
    gradient_accumulation_steps=2,    # Reduced since we have larger batch
    gradient_checkpointing=False,     # Disabled for speed on GB10
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    max_steps=200,
    save_strategy="no",
    logging_steps=1,
    output_dir=new_model,
    optim="adamw_torch",  # Standard AdamW on GB10 (no paged optimizer needed)
    warmup_steps=10,
    bf16=True,
    report_to="none",  # Change to "wandb" if using W&B
)

In [ ]:
dpo_trainer = DPOTrainer(
    model,
    ref_model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    beta=0.1,
    max_prompt_length=512,
    max_length=1024, 
    force_use_ref_model=True
)

## 5. Train with DPO

In [ ]:
# Fine-tune model with DPO
dpo_trainer.train()

## 6. Save and Test Model

In [ ]:
# Save the LoRA adapter
dpo_trainer.model.save_pretrained("final_ckpt")
tokenizer.save_pretrained("final_ckpt")
print("LoRA adapter saved to final_ckpt/")

In [ ]:
# Flush memory
del dpo_trainer, model, ref_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Reload model in BF16 for merging
base_model_reload = AutoModelForCausalLM.from_pretrained(
    "allura-forge/Llama-3.3-8B-Instruct",
    token=os.environ["HF_KEY"], 
    cache_dir="/project/models",
    return_dict=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("allura-forge/Llama-3.3-8B-Instruct", token=os.environ["HF_KEY"])
tokenizer.chat_template = chat_template

In [ ]:
# Merge base model with the LoRA adapter
model = PeftModel.from_pretrained(base_model_reload, "final_ckpt")
model = model.merge_and_unload()

# Save merged model and tokenizer
tokenizer.save_pretrained(new_model)
model.save_pretrained(new_model)
print(f"Merged model saved to: {new_model}")

In [ ]:
# Create pipeline for testing
pipeline = transformers.pipeline(
    "text-generation",
    model=new_model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
# Test the fine-tuned model
message = [
    {"role": "system", "content": "You are a helpful assistant chatbot that provides concise answers."},
    {"role": "user", "content": "What are GPUs and why would I use them for machine learning tasks?"}
]

tokenizer_test = AutoTokenizer.from_pretrained(new_model)
prompt = tokenizer_test.apply_chat_template(message, add_generation_prompt=True, tokenize=False)

# Generate response
sequences = pipeline(
    prompt,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    num_return_sequences=1,
    max_length=500,
)

print("Generated response:")
print(sequences[0]['generated_text'][len(prompt):])